In [17]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import math

In [2]:
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

In [3]:
print(X_train)

[[2104    5    1   45]
 [1416    3    2   40]
 [ 852    2    1   35]]


# Multiple Regression
For this case we have multiple features and a single target vector. We want to utilize all the features available to get the target's esimation.

$X$ is a matrix containing $m$ examples and $n$ features. So, it is a matrix with $(m,n)$ dimensions

$X = $ $\begin{pmatrix} x_{0}^{(0)} & x_{1}^{(0)} & \ldots & x_0^{(n-1)} \\ x_{0}^{(1)} & x_{1}^{(1)} & \ldots & x_0^{(n-1)} \\ \ldots \\ x_0^{(m-1)} & x_1^{(m-1)} & \ldots & x_{n-1}^{(m-1)} \end{pmatrix}$

Now for $w$ and $b$, we will set them to some initial value. We will set it close to the optimal value i.e

In [4]:
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])
print(f"w_init shape = {w_init.shape}, b_init = {b_init}")

w_init shape = (4,), b_init = 785.1811367994083


$f_{w,b}(x) = w_0x_0 + w_1x_1 + \ldots + w_nx_n + b$

Or in vector form, $f_{w,b}(x) = w \cdot x + b$

Now, using the power of vectorization, we completely skip multiplying w and x row by row in a loop and instead use their dot product which is given by numpy

In [5]:
def predict(x, w, b):
    p = np.dot(x, w) + b
    return p

In [6]:
x_vec = X_train[0,:]
print(x_vec)
f_wb = predict(x_vec, w_init, b_init)
print(f_wb)

[2104    5    1   45]
459.9999976194083


In [7]:
def compute_cost(X, y, w, b):
    cost = 0.0
    m = X.shape[0]
    for i in range(m):
        f_wb_i = np.dot(X[i], w) + b
        cost += (f_wb_i - y[i])**2
    cost = cost/(2 * m)
    return cost

In [8]:
cost = compute_cost(X_train, y_train, w_init, b_init)
print(f"Cost at near optimal w and b is: {cost}")

Cost at near optimal w and b is: 1.5578904428966628e-12


## Computing Gradient
Now, for multiple features, we compute the gradients as follows:

$ \frac{\partial{J(w,b)}}{\partial{w_j}} = \frac{\partial}{\partial{w_j}} (\frac{1}{2m} \sum\limits_{i = 0}^{m - 1} (f_{w,b}x^{(i)} - y^{(i)})^2)$

$ \frac{\partial{J(w,b)}}{\partial{w_j}} =  \frac{1}{m} \sum\limits_{i = 0}^{m - 1} (f_{w,b}x^{(i)} - y^{(i)}) * x_{j}^{(i)}$

Similarly,

$ \frac{\partial{J(w,b)}}{\partial{b}} = \frac{\partial}{\partial{b}} (\frac{1}{2m} \sum\limits_{i = 0}^{m - 1} (f_{w,b}x^{(i)} - y^{(i)})^2)$

$ \frac{\partial{J(w,b)}}{\partial{w_j}} = \frac{1}{m} \sum\limits_{i = 0}^{m - 1} (f_{w,b}x^{(i)} - y^{(i)})$


In [9]:
def compute_gradient(X, y, w, b):
    m, n = X.shape
    dj_dw = np.zeros(n,)
    dj_db = 0

    for i in range(m):
        err = (np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err * X[i, j]
        dj_db = dj_db + err
    dj_dw = dj_dw / m
    dj_db = dj_db / m
    return dj_db, dj_dw

In [42]:
tmp_dj_dw, tmp_dj_db = compute_gradient(X_train, y_train, w_init, b_init)
print(f"Gradiant for Wj :{tmp_dj_dw}, Gradiant for b :{tmp_dj_db}")


Gradiant for Wj :-1.6739251501955248e-06, Gradiant for b :[-2.72623577e-03 -6.27197263e-06 -2.21745578e-06 -6.92403391e-05]


Now we compute the cost using the cost function but this will be for multiple features. So, we tweak our previous cost function a little

In [25]:
def gradient_descent(X, y, w_in, b_in, iterations, alpha, gradient_function, cost_function):
    J_hist = []
    w = copy.deepcopy(w_in)
    b = b_in
    for i in range(iterations):
        dj_db, dj_dw = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        if(i < 10000):
            J_hist.append(cost_function(X,y,w,b))
        if i % math.ceil(iterations/10)==0 :
            print(f"Iterations {i}: Cost {J_hist[-1]} ")
    return w, b, J_hist

In [26]:
initial_w = np.zeros_like(w_init)
initial_b = 0.0

iterations = 1000
alpha = 5.0e-7
w_final, b_final, J_hist = gradient_descent(X_train, y_train, initial_w, initial_b, iterations, alpha, compute_cost, compute_gradient)
m, _ = X_train.shape
for i in range(m):
    print(f"Prediction: {np.dot(X_train[i], w_final) + b_final:0.2f}, Target Value: {y_train[i]}")

Iterations 0: Cost (np.float64(61.94870633333326), array([93505.18809822,   208.59807378,    98.60653178,  2511.63663444])) 
Iterations 100: Cost (np.float64(4.807203791772726), array([ -3.66320413,  -6.58614215,  22.95467142, 144.49607966])) 
Iterations 200: Cost (np.float64(4.784514112973786), array([ -3.64685733,  -6.58151526,  22.92160911, 143.85134066])) 
Iterations 300: Cost (np.float64(4.761925121042897), array([ -3.63058306,  -6.57690861,  22.88869267, 143.20946232])) 
Iterations 400: Cost (np.float64(4.739436369247206), array([ -3.614381  ,  -6.57232211,  22.85592146, 142.57043194])) 
Iterations 500: Cost (np.float64(4.717047412835797), array([ -3.59825083,  -6.56775566,  22.82329484, 141.93423688])) 
Iterations 600: Cost (np.float64(4.694757809031226), array([ -3.58219222,  -6.56320919,  22.79081217, 141.30086457])) 
Iterations 700: Cost (np.float64(4.672567117020283), array([ -3.56620486,  -6.55868259,  22.7584728 , 140.67030248])) 
Iterations 800: Cost (np.float64(4.6504748